# 06 — Finalna Evaluacija i Poređenje Svih Modela (PyTorch)
**Cilj:** Skupiti rezultate svih modela iz prethodnih notebooka, napraviti detaljnu komparativnu analizu i izvući zaključke.

**Modeli koji se porede:**
1. MLP + Class Weight (baseline)
2. MLP + SMOTE (baseline)
3. MLP + SMOTE+Under (baseline)
4. Deep BN MLP + Focal Loss
5. ResNet MLP + Focal Loss
6. Tuned MLP (Optuna Bayesian Optimization)
7. Autoencoder (unsupervised)

> **Framework:** Ovaj notebook koristi **PyTorch** umesto TensorFlow/Keras. Modeli su sačuvani kao `.pt` fajlovi (PyTorch `state_dict` format).

---

## 0. Imports

Ovaj notebook je evaluacioni — nema treniranja. Koristimo samo PyTorch za učitavanje sačuvanih modela i sklearn za metrike.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import json, os, warnings

import torch
import torch.nn as nn

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
    roc_curve, precision_recall_curve
)

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PROC    = '../data/processed/'
MODELS  = '../models/'
RESULTS = '../results/'

print(f'PyTorch: {torch.__version__} | Uređaj: {DEVICE}')
print('✅ Setup OK')

## 1. Definicija arhitektura

**Zašto ponovo definišemo klase?**

PyTorch čuva modele kao `state_dict` — rečnik težina, **bez arhitekture**. Za razliku od Kerasa (koji čuva i arhitekturu i težine u `.keras` fajlu), PyTorch zahteva da ponovo definišemo mrežne klase pre učitavanja težina.

Ovo je dizajnerska odluka PyTorcha: arhitektura ostaje u Python kodu (eksplicitna kontrola), a ne u binarnom fajlu (opaque format).

Kopiramo definicije klasa iz prethodnih notebooka:

In [ ]:
# ====== Baseline MLP ======
class BaselineMLP(nn.Module):
    def __init__(self, input_dim, hidden_units=(64, 32, 16), dropout_rate=0.3):
        super().__init__()
        layers_list, in_f = [], input_dim
        for units in hidden_units:
            layers_list.append(nn.Sequential(
                nn.Linear(in_f, units), nn.ReLU(), nn.Dropout(p=dropout_rate)
            ))
            in_f = units
        self.hidden = nn.ModuleList(layers_list)
        self.output = nn.Linear(in_f, 1)

    def forward(self, x):
        for layer in self.hidden:
            x = layer(x)
        return self.output(x)


# ====== Deep BN MLP ======
class DeepBNMLP(nn.Module):
    def __init__(self, input_dim, hidden_units=(128, 64, 32, 16), dropout_rate=0.3):
        super().__init__()
        layers_list, in_f = [], input_dim
        for units in hidden_units:
            layers_list += [
                nn.Linear(in_f, units), nn.BatchNorm1d(units), nn.ReLU(), nn.Dropout(p=dropout_rate)
            ]
            in_f = units
        self.backbone = nn.Sequential(*layers_list)
        self.output   = nn.Linear(in_f, 1)

    def forward(self, x):
        return self.output(self.backbone(x))


# ====== ResNet MLP ======
class ResidualBlock(nn.Module):
    def __init__(self, in_features, out_features, dropout_rate=0.3):
        super().__init__()
        self.main_path = nn.Sequential(
            nn.Linear(in_features, out_features), nn.BatchNorm1d(out_features),
            nn.ReLU(), nn.Dropout(p=dropout_rate),
            nn.Linear(out_features, out_features), nn.BatchNorm1d(out_features)
        )
        self.skip    = nn.Linear(in_features, out_features, bias=False) \
                       if in_features != out_features else nn.Identity()
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x):
        return self.dropout(self.relu(self.main_path(x) + self.skip(x)))


class ResNetMLP(nn.Module):
    def __init__(self, input_dim, hidden_units=(128, 64, 32), dropout_rate=0.3):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_units[0]), nn.BatchNorm1d(hidden_units[0]), nn.ReLU()
        )
        self.blocks = nn.Sequential(*[
            ResidualBlock(in_f, out_f, dropout_rate)
            for in_f, out_f in zip(hidden_units[:-1], hidden_units[1:])
        ])
        self.output = nn.Linear(hidden_units[-1], 1)

    def forward(self, x):
        return self.output(self.blocks(self.input_proj(x)))


# ====== Autoencoder ======
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_units=(64, 32, 16), encoding_dim=8, dropout_rate=0.2):
        super().__init__()
        layers_list, in_f = [], input_dim
        for units in hidden_units:
            layers_list += [nn.Linear(in_f, units), nn.BatchNorm1d(units),
                            nn.ReLU(), nn.Dropout(p=dropout_rate)]
            in_f = units
        layers_list.append(nn.Linear(in_f, encoding_dim))
        self.network = nn.Sequential(*layers_list)

    def forward(self, x):
        return self.network(x)


class Decoder(nn.Module):
    def __init__(self, encoding_dim=8, hidden_units=(16, 32, 64), output_dim=31, dropout_rate=0.2):
        super().__init__()
        layers_list, in_f = [], encoding_dim
        for units in hidden_units:
            layers_list += [nn.Linear(in_f, units), nn.BatchNorm1d(units),
                            nn.ReLU(), nn.Dropout(p=dropout_rate)]
            in_f = units
        layers_list.append(nn.Linear(in_f, output_dim))
        self.network = nn.Sequential(*layers_list)

    def forward(self, z):
        return self.network(z)


class Autoencoder(nn.Module):
    def __init__(self, input_dim, encoding_dim=8, hidden_units=(64, 32, 16), dropout_rate=0.2):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_units, encoding_dim, dropout_rate)
        self.decoder = Decoder(encoding_dim, tuple(reversed(hidden_units)), input_dim, dropout_rate)

    def forward(self, x):
        return self.decoder(self.encoder(x))


print('✅ Sve arhitekture definisane')

## 2. Učitavanje podataka i modela

**Učitavanje PyTorch modela iz `.pt` fajlova:**
1. Instanciramo prazan model (arhitektura sa slučajnim težinama)
2. `model.load_state_dict(torch.load(...))` — učitavamo sačuvane težine
3. `model.eval()` — postavljamo model u evaluation mode (isključuje Dropout i BN trening statistike)
4. `.to(DEVICE)` — prebacujemo na GPU ako je dostupan

Za Tuned MLP arhitektura zavisi od Optuna pretrage — učitavamo posebno sačuvane hiperparametre iz JSON-a.

In [ ]:
X_val  = np.load(f'{PROC}X_val.npy')
X_test = np.load(f'{PROC}X_test.npy')
y_val  = np.load(f'{PROC}y_val.npy')
y_test = np.load(f'{PROC}y_test.npy')

INPUT_DIM = X_test.shape[1]

X_val_t  = torch.tensor(X_val,  dtype=torch.float32).to(DEVICE)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)

print(f'Test set: {X_test.shape} | Fraud: {y_test.sum()} ({y_test.mean()*100:.2f}%)')

In [ ]:
def load_pt_model(model_instance, filename):
    """
    Učitava sačuvane PyTorch težine u datu instancu modela.
    Vraća model u eval modu na DEVICE-u, ili None ako fajl ne postoji.
    """
    path = f'{MODELS}{filename}'
    if not os.path.exists(path):
        print(f'⚠️  Nije pronađen: {filename}')
        return None
    state = torch.load(path, map_location=DEVICE)
    model_instance.load_state_dict(state)
    model_instance.eval()
    return model_instance.to(DEVICE)


# Učitavamo sve modele
models = {}

for name, instance, fname in [
    ('MLP + Class Weight',  BaselineMLP(INPUT_DIM),       'mlp_classweight.pt'),
    ('MLP + SMOTE',         BaselineMLP(INPUT_DIM),       'mlp_smote.pt'),
    ('MLP + SMOTE+Under',   BaselineMLP(INPUT_DIM),       'mlp_combined.pt'),
    ('Deep BN MLP + Focal', DeepBNMLP(INPUT_DIM),         'mlp_bn.pt'),
    ('ResNet MLP + Focal',  ResNetMLP(INPUT_DIM),         'mlp_resnet.pt'),
    ('Autoencoder',         Autoencoder(INPUT_DIM),       'autoencoder.pt'),
]:
    m = load_pt_model(instance, fname)
    if m is not None:
        models[name] = m
        print(f'✅ Učitan: {name}')

# Tuned MLP — arhitektura zavisi od Optuna pretrage
tuned_hp_path = f'{MODELS}mlp_tuned_hp.json'
if os.path.exists(tuned_hp_path):
    with open(tuned_hp_path) as f:
        bp = json.load(f)
    hidden = [bp[f'units_{i}'] for i in range(bp['n_layers'])]
    layers_list, in_f = [], INPUT_DIM
    for units in hidden:
        layers_list.append(nn.Linear(in_f, units))
        if bp.get('use_bn', False):
            layers_list.append(nn.BatchNorm1d(units))
        layers_list += [nn.ReLU(), nn.Dropout(p=bp['dropout'])]
        in_f = units
    layers_list.append(nn.Linear(in_f, 1))
    tuned_model = nn.Sequential(*layers_list)
    m = load_pt_model(tuned_model, 'mlp_tuned.pt')
    if m is not None:
        models['Tuned MLP (Optuna)'] = m
        print('✅ Učitan: Tuned MLP (Optuna)')
else:
    print('⚠️  mlp_tuned_hp.json nije pronađen — preskačem Tuned MLP')

## 3. Evaluacija svih modela

Definišemo helper funkcije za uniformnu evaluaciju, prilagođene PyTorchu:

- `get_scores` — za supervised modele vraća sigmoid verovatnoće (logiti → sigmoid), za autoencoder vraća MSE rekonstrukcijsku grešku (veći score = verovatnije anomalija)
- `best_threshold_f1` — pretraga optimalnog praga po F1 na validation setu

Sve predikcije se rade u `torch.no_grad()` bloku za maksimalnu efikasnost.

In [ ]:
def get_scores(model, X_tensor, is_autoencoder=False):
    """Vraća anomaly/probability scores kao NumPy niz."""
    model.eval()
    with torch.no_grad():
        if is_autoencoder:
            X_rec = model(X_tensor)
            X_np  = X_tensor.cpu().numpy()
            X_rec_np = X_rec.cpu().numpy()
            return np.mean(np.power(X_np - X_rec_np, 2), axis=1)
        else:
            logits = model(X_tensor)
            return torch.sigmoid(logits).cpu().numpy().ravel()


def best_threshold_f1(scores, y_true):
    """Optimalni threshold po F1 na validation setu."""
    ths = np.linspace(scores.min(), scores.max(), 200)
    f1s = [f1_score(y_true, (scores >= t).astype(int)) for t in ths]
    return ths[np.argmax(f1s)]


all_results = []
all_curves  = {}

for name, model in models.items():
    is_ae = 'Autoencoder' in name

    val_scores  = get_scores(model, X_val_t,  is_ae)
    test_scores = get_scores(model, X_test_t, is_ae)

    t      = best_threshold_f1(val_scores, y_val)
    y_pred = (test_scores >= t).astype(int)

    roc_auc = roc_auc_score(y_test, test_scores)
    pr_auc  = average_precision_score(y_test, test_scores)
    f1      = f1_score(y_test, y_pred)
    prec    = precision_score(y_test, y_pred, zero_division=0)
    rec     = recall_score(y_test, y_pred)

    fpr, tpr, _ = roc_curve(y_test, test_scores)
    p, r, _     = precision_recall_curve(y_test, test_scores)
    all_curves[name] = {'fpr': fpr, 'tpr': tpr, 'prec': p, 'rec': r}

    all_results.append({
        'Model': name, 'ROC-AUC': round(roc_auc, 4), 'PR-AUC': round(pr_auc, 4),
        'F1': round(f1, 4), 'Precision': round(prec, 4), 'Recall': round(rec, 4),
        'Threshold': round(t, 4), 'Tip': 'Unsupervised' if is_ae else 'Supervised'
    })
    print(f'{name:<30} ROC={roc_auc:.4f}  PR={pr_auc:.4f}  F1={f1:.4f}  Rec={rec:.4f}')

results_df = pd.DataFrame(all_results).set_index('Model')

## 4. Sumarni pregled metrika

Prikazujemo kompletnu tabelu metrika sa color-coding-om — zelena ćelija označava maksimalnu vrednost za svaku metriku. Ovo nam odmah daje jasan pregled koji model dominira po kojoj dimenziji performansi.

In [ ]:
print('\n=== SUMARNI PREGLED SVIH MODELA ===')
display(
    results_df[['ROC-AUC', 'PR-AUC', 'F1', 'Precision', 'Recall', 'Tip']]
    .sort_values('PR-AUC', ascending=False)
    .style.highlight_max(
        color='#90EE90',
        subset=['ROC-AUC', 'PR-AUC', 'F1', 'Precision', 'Recall']
    )
)

results_df.to_csv(f'{RESULTS}06_all_results.csv')

## 5. Vizualizacija — ROC i Precision-Recall krive

**ROC kriva** prikazuje trade-off između True Positive Rate (TPR = Recall) i False Positive Rate (FPR) za različite threshold-ove. Idealna kriva ide ka gornjem levom uglu.

**PR kriva** prikazuje trade-off između Precision i Recall. Za nebalansirane podatke, ovo je informativnija metrika — dijagonala (random baseline) je na nivou fraud frakcije (~0.17%), ne na 0.5 kao kod ROC-a.

Sve krive su prikazane zajedno radi direktnog poređenja.

In [ ]:
colors = plt.cm.tab10(np.linspace(0, 1, len(all_curves)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for (name, curves), c in zip(all_curves.items(), colors):
    auc_val = results_df.loc[name, 'ROC-AUC']
    axes[0].plot(curves['fpr'], curves['tpr'], lw=1.8, color=c,
                 label=f"{name} ({auc_val:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Krive — Svi Modeli', fontweight='bold')
axes[0].legend(fontsize=7, loc='lower right')

for (name, curves), c in zip(all_curves.items(), colors):
    pr_val = results_df.loc[name, 'PR-AUC']
    axes[1].plot(curves['rec'], curves['prec'], lw=1.8, color=c,
                 label=f"{name} ({pr_val:.3f})")

axes[1].axhline(y_test.mean(), color='black', linestyle='--', alpha=0.4,
                label=f'Random baseline ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Krive — Svi Modeli', fontweight='bold')
axes[1].legend(fontsize=7, loc='upper right')

plt.tight_layout()
plt.savefig(f'{RESULTS}06_roc_pr_curves.png', bbox_inches='tight')
plt.show()

## 6. Confusion Matrice

Confusion matrix prikazuje detaljnu sliku predikcija:
- **TP (True Positive)** — ispravno detektovane prevare ✅
- **TN (True Negative)** — ispravno propuštene legitimne ✅  
- **FP (False Positive)** — lažni alarm (legitimna označena kao fraud) ⚠️
- **FN (False Negative)** — propuštena prevara ❌ ← najkritičniji tip greške

Svaki model ima optimizovani threshold, što znači da confusion matrice nisu direktno uporedive — ali daju uvid u konkretne brojeve.

In [ ]:
n_models = len(models)
cols     = 3
rows     = (n_models + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes_flat = axes.ravel()

for i, (name, model) in enumerate(models.items()):
    is_ae   = 'Autoencoder' in name
    scores  = get_scores(model, X_test_t, is_ae)
    val_sc  = get_scores(model, X_val_t,  is_ae)
    t       = best_threshold_f1(val_sc, y_val)
    y_pred  = (scores >= t).astype(int)

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes_flat[i],
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    axes_flat[i].set_title(name, fontsize=9, fontweight='bold')
    axes_flat[i].set_ylabel('Stvarno')
    axes_flat[i].set_xlabel('Predviđeno')

# Skrivamo prazne subplot-ove
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Confusion Matrice — Svi Modeli', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS}06_confusion_matrices.png', bbox_inches='tight')
plt.show()

## 7. Analiza Cost-Benefit (Poslovni kontekst)

Metrike poput F1 i PR-AUC su važne za ML evaluaciju, ali poslovni donosioci odluka razumeju **novac**.

**Pretpostavke troškova:**
- `COST_FN = 200€` — propuštena prevara (False Negative): direktni finansijski gubitak
- `COST_FP = 10€` — lažni alarm (False Positive): operativni trošak ručne provere, nezadovoljstvo korisnika

**Baseline scenarij** (bez detekcije): svi fraudovi su propušteni → trošak = `n_fraud × COST_FN`.

**Uštedna** = Baseline - Ukupan trošak modela.

Ova analiza nam pomaže da odaberemo model koji je ne samo statistički bolji, već i **ekonomski opravdan**.

In [ ]:
COST_FN = 200  # € — propuštena prevara
COST_FP = 10   # € — lažni alarm

print(f'=== COST-BENEFIT ANALIZA ===')
print(f'Pretpostavke: FN trošak = {COST_FN}€, FP trošak = {COST_FP}€')

cost_results = []
n_fraud = int(y_test.sum())
n_legit = int((y_test == 0).sum())

for name, model in models.items():
    is_ae  = 'Autoencoder' in name
    scores = get_scores(model, X_test_t, is_ae)
    val_sc = get_scores(model, X_val_t,  is_ae)
    t      = best_threshold_f1(val_sc, y_val)
    y_pred = (scores >= t).astype(int)

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    total_cost    = fn * COST_FN + fp * COST_FP
    baseline_cost = n_fraud * COST_FN
    savings       = baseline_cost - total_cost

    cost_results.append({
        'Model': name, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'FN Cost (€)': fn * COST_FN,
        'FP Cost (€)': fp * COST_FP,
        'Total Cost (€)': total_cost,
        'Savings vs Baseline (€)': savings
    })

cost_df = pd.DataFrame(cost_results).set_index('Model')
display(cost_df.sort_values('Total Cost (€)')
        .style.highlight_min(color='#90EE90', subset=['Total Cost (€)'])
              .highlight_max(color='#90EE90', subset=['Savings vs Baseline (€)']))

fig, ax = plt.subplots(figsize=(12, 5))
sorted_cost = cost_df.sort_values('Total Cost (€)', ascending=True)
bars = ax.barh(sorted_cost.index, sorted_cost['Total Cost (€)'],
               color='salmon', edgecolor='black')
ax.axvline(n_fraud * COST_FN, color='red', linestyle='--', lw=2,
           label=f'Baseline (nema detekcije) = {n_fraud * COST_FN:,}€')
for bar, val in zip(bars, sorted_cost['Total Cost (€)']):
    ax.text(val + 200, bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f}€', va='center', fontsize=9)
ax.set_xlabel('Ukupan Trošak (€)')
ax.set_title('Cost-Benefit Analiza — Ukupan Trošak po Modelu', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS}06_cost_benefit.png', bbox_inches='tight')
plt.show()

## 8. Zaključci i Preporuke

Na osnovu svih eksperimenata izvlačimo ključne zaključke o modelima, strategijama tretmana nebalansa i preporukama za produkciju.

In [ ]:
best_pr  = results_df['PR-AUC'].idxmax()
best_f1  = results_df['F1'].idxmax()
best_rec = results_df['Recall'].idxmax()

print('=' * 65)
print('FINALNI ZAKLJUČCI')
print('=' * 65)
print()
print('1. KLJUČNE METRIKE za imbalanced fraud detection:')
print('   PR-AUC > F1 > Recall — ROC-AUC je manje informativan')
print('   zbog ekstremne nebalansiranosti (0.17% fraud)')
print()
print(f'2. BEST PR-AUC:   {best_pr} ({results_df.loc[best_pr,"PR-AUC"]:.4f})')
print(f'   BEST F1-Score: {best_f1} ({results_df.loc[best_f1,"F1"]:.4f})')
print(f'   BEST Recall:   {best_rec} ({results_df.loc[best_rec,"Recall"]:.4f})')
print()
print('3. PYTORCH SPECIFIČNOSTI:')
print('   - Ručni training loop = veća kontrola nad treningom')
print('   - state_dict format = eksplicitno upravljanje težinama')
print('   - Optuna (umesto Keras Tuner) za HP pretragu')
print()
print('4. TRETMAN NEBALANSIRANOSTI:')
print('   - BCEWithLogitsLoss(pos_weight) = class_weight ekvivalent')
print('   - SMOTE pomaže ali rizikuje overfitting na sintetičkim podacima')
print('   - Focal Loss značajno poboljšava Recall')
print()
print('5. ARHITEKTURA:')
print('   - BatchNorm1d ubrzava trening i stabilizuje (train/eval mode!)')
print('   - Residual connections pomažu kod dupljih mreža')
print('   - Optuna Bayesian Opt. efikasniji od grid search-a')
print()
print('6. AUTOENCODER (unsupervised):')
print('   - Treba samo legitimne uzorke za trening')
print('   - MSE rekonstrukcijska greška = anomaly score')
print('   - Slabiji Recall od supervised modela, ali komplementaran')
print()
print('7. PREPORUKA za produkciju:')
print(f'   → {best_pr}')
print('   → Optimizovati threshold po Recall (ne F1!) za max detekciju')
print('   → Razmotriti ensemble: Supervised + Autoencoder score')
print('=' * 65)

## 9. Finalni Summary Dashboard

Kompletni dashboard sa svim ključnim metrikama i vizualizacijama na jednom mestu — pogodan za prezentaciju rezultata.

In [ ]:
fig = plt.figure(figsize=(20, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# PR-AUC ranking
ax1 = fig.add_subplot(gs[0, 0])
sorted_pr  = results_df['PR-AUC'].sort_values(ascending=True)
colors_bar = ['#e74c3c' if v == sorted_pr.max() else '#3498db' for v in sorted_pr]
ax1.barh(sorted_pr.index, sorted_pr, color=colors_bar, edgecolor='black')
ax1.set_title('PR-AUC Ranking', fontweight='bold')
ax1.set_xlim(0.6, 1.0)
ax1.tick_params(labelsize=8)
for i, v in enumerate(sorted_pr):
    ax1.text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=8)

# F1 ranking
ax2 = fig.add_subplot(gs[0, 1])
sorted_f1  = results_df['F1'].sort_values(ascending=True)
colors_f1  = ['#e74c3c' if v == sorted_f1.max() else '#2ecc71' for v in sorted_f1]
ax2.barh(sorted_f1.index, sorted_f1, color=colors_f1, edgecolor='black')
ax2.set_title('F1 Ranking', fontweight='bold')
ax2.set_xlim(0.6, 1.0)
ax2.tick_params(labelsize=8)
for i, v in enumerate(sorted_f1):
    ax2.text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=8)

# Recall ranking
ax3 = fig.add_subplot(gs[0, 2])
sorted_rec = results_df['Recall'].sort_values(ascending=True)
colors_rec = ['#e74c3c' if v == sorted_rec.max() else '#e67e22' for v in sorted_rec]
ax3.barh(sorted_rec.index, sorted_rec, color=colors_rec, edgecolor='black')
ax3.set_title('Recall Ranking', fontweight='bold')
ax3.set_xlim(0.6, 1.0)
ax3.tick_params(labelsize=8)
for i, v in enumerate(sorted_rec):
    ax3.text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=8)

# PR krive
ax4 = fig.add_subplot(gs[1, 0:2])
line_colors = plt.cm.tab10(np.linspace(0, 1, len(all_curves)))
for (name, curves), c in zip(all_curves.items(), line_colors):
    short_name = name.split('+')[0].strip()
    ax4.plot(curves['rec'], curves['prec'], lw=1.5, color=c,
             label=f"{short_name} ({results_df.loc[name,'PR-AUC']:.3f})")
ax4.axhline(y_test.mean(), color='black', linestyle='--', alpha=0.4, label='Random baseline')
ax4.set_xlabel('Recall')
ax4.set_ylabel('Precision')
ax4.set_title('Precision-Recall Krive', fontweight='bold')
ax4.legend(fontsize=7, loc='upper right')

# Tabela metrika
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
tbl_data = results_df[['PR-AUC', 'F1', 'Recall']].sort_values('PR-AUC', ascending=False)
tbl = ax5.table(
    cellText=tbl_data.values.round(4),
    rowLabels=[n[:20] for n in tbl_data.index],
    colLabels=tbl_data.columns,
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1.2, 1.5)
ax5.set_title('Tabela Metrika', fontweight='bold', pad=15)

fig.suptitle('Credit Card Fraud Detection — Finalni Dashboard (PyTorch)',
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig(f'{RESULTS}06_final_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n✅ PROJEKAT ZAVRŠEN!')
print(f'Svi rezultati sačuvani u: {RESULTS}')
print(f'Svi modeli sačuvani u:    {MODELS}')